# Session 6 — Tuning: where the compute budget goes

Session 5 left two loose ends and said so. The number of trees for boosting came from reading a
validation curve by eye, and the learning rate came from trying three values in a table. Every other
setting was left at whatever the library ships.

This session is about doing that properly: stating a **search space**, spending a **budget** in it,
and reporting the result honestly. The question it answers is not "how do I run a search" — that is
three lines of code — but **when a search is worth running at all**.

## Learning objectives

By the end of this session you will be able to:

1. State what a tuning procedure optimizes, and on which data, without touching the test set.
2. Run a **random search**, and explain why sampling from distributions covers a space better than
   trying every combination of a few listed values.
3. Explain why **the search space is the part that carries your knowledge of the model**, and what a
   badly chosen space costs.
4. Use **Optuna**: a study, a trial, the **TPE** sampler, and **pruning** to stop hopeless trials
   early.
5. Read the plots a study produces, including which hyperparameters mattered.
6. Use **nested cross-validation** to estimate what a tuning *procedure* achieves, and explain why
   the search's own best score is optimistic.
7. Decide when tuning is worth the compute — and recognise the cases where it buys nothing.

## How the session is organized

**Part 1** works on the churn data from sessions 4 and 5, where the models are already familiar.
**Part 2** runs the same procedure end to end on the Ames house prices from sessions 2 and 3, to see
whether the conclusions hold when the task is regression rather than classification.

> **A warning about time.** This notebook fits thousands of models. Every search here is deliberately
> small — a few dozen candidates — and the whole notebook takes about four minutes. Real tuning runs
> are left going for hours.

The cell below imports everything the notebook uses and fixes the seed. Optuna's own progress
messages are left switched on, so once a study starts you will see one line per trial, including the
ones that get stopped early:

In [ ]:
import time

import numpy as np
import optuna
import pandas as pd
import xgboost as xgb
from optuna import visualization as optuna_plots
from optuna_integration import XGBoostPruningCallback
from scipy.stats import loguniform, randint, uniform
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import GradientBoostingClassifier, GradientBoostingRegressor
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression, Ridge
from sklearn.metrics import roc_auc_score, root_mean_squared_error
from sklearn.model_selection import (
    RandomizedSearchCV,
    cross_val_score,
    train_test_split,
)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.tree import DecisionTreeClassifier

RANDOM_STATE = 42
# optuna.logging.set_verbosity(optuna.logging.WARNING)

## What a tuning procedure optimizes

Session 3 separated two kinds of number inside a model.

- A **parameter** is learned by `fit`: the weights of a logistic regression, the split points of a
  tree. You never set these by hand.
- A **hyperparameter** is fixed **before** `fit` and controls how the fitting happens: a tree's
  `max_depth`, KNN's $k$, boosting's learning rate. No amount of training data sets them for you.

**Tuning** is a search over hyperparameters. Three things have to be stated before it can run:

- **the search space** — which hyperparameters, and which values each may take;
- **the objective** — the number the search maximises or minimises;
- **the budget** — how many candidates may be evaluated.

The objective is where tuning goes wrong most often. It must be an **honest estimate**, because the
search will find and exploit anything that is not. Scoring candidates on the training data selects the
model that memorises best. Scoring them on the **test** set is worse: the test set stops being unseen
data the moment it chooses anything, and the final number you report becomes a fiction.

So the objective is a **cross-validated score on the training data**, exactly as in session 3, and the
test set is opened once, at the end, to confirm.

The cell below loads the churn data and rebuilds the preparation from sessions 4 and 5. Nothing here
is new; check that the shapes and the churn rate match what you saw there:

In [ ]:
churn = pd.read_csv("data/telco_churn.csv").drop(columns="customerID")
churn["TotalCharges"] = pd.to_numeric(churn["TotalCharges"], errors="coerce").fillna(0.0)
churn["Churn"] = (churn["Churn"] == "Yes").astype(int)

X = churn.drop(columns="Churn")
y = churn["Churn"]
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, stratify=y, random_state=RANDOM_STATE
)

NUMERIC_COLS = ["tenure", "MonthlyCharges", "TotalCharges"]
CATEGORICAL_COLS = [column for column in X.columns if column not in NUMERIC_COLS]
preprocessor = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), NUMERIC_COLS),
        ("cat", OneHotEncoder(handle_unknown="ignore", drop="if_binary", sparse_output=False),
         CATEGORICAL_COLS),
    ],
    verbose_feature_names_out=False,
).set_output(transform="pandas")


def make_model(estimator) -> Pipeline:
    """Chain the shared preprocessing recipe to an estimator.

    Args:
        estimator: Any scikit-learn classifier.

    Returns:
        A pipeline that prepares the columns and then applies the estimator.
    """
    return Pipeline([("prep", preprocessor), ("model", estimator)])


print(f"training rows: {len(X_train)}   test rows: {len(X_test)}")
print(f"churn rate: train {y_train.mean():.4f}, test {y_test.mean():.4f}")

### The two models we start from

Session 5 measured nine candidates. Two of them are all we need here, and they sit at opposite ends of
the same scale.

- **An unrestricted decision tree.** Left at its defaults it grows until its leaves are nearly pure,
  which memorises the training data. It scored **0.6594**, the worst real model in that session.
- **Gradient boosting with 45 trees.** The number of trees came from a validation curve, and every
  other setting was a default. It scored **0.8464**, the best.

The cell below re-measures both, so that the rest of the session compares against numbers produced
here rather than quoted from elsewhere:

In [ ]:
def cv_auc(estimator, label: str) -> dict:
    """Cross-validate one estimator on the churn training data.

    Args:
        estimator: Any scikit-learn classifier, without preprocessing.
        label: Name to report the result under.

    Returns:
        The label, the mean ROC AUC across five folds and its standard deviation.
    """
    scores = cross_val_score(
        make_model(estimator), X_train, y_train, cv=5, scoring="roc_auc", n_jobs=-1
    )
    return {"model": label, "CV AUC": scores.mean(), "sd": scores.std()}


starting_points = pd.DataFrame(
    [
        cv_auc(DecisionTreeClassifier(random_state=RANDOM_STATE), "Decision tree (defaults)"),
        cv_auc(
            GradientBoostingClassifier(n_estimators=45, random_state=RANDOM_STATE),
            "Gradient boosting (45 trees)",
        ),
    ]
)
starting_points.round(4)

**0.6594 and 0.8464**, as in session 5. Keep both in view: the session's real question is how much
tuning can move each of them.

## Three ways to choose the next candidate

A search needs a rule for deciding which candidate to try next, and there are three families of rule.
All three are used in this session, so it is worth seeing them side by side before any of them runs.

**Random search** takes a **distribution** for each hyperparameter and draws candidates from it, as
many as the budget allows. Every draw is a new value, and the budget is yours to choose — it does not
grow with the number of hyperparameters.

**Adaptive search** — the family Optuna belongs to — treats tuning as an **optimization problem**.
It fits a model of the objective as it goes: each new candidate is chosen using what the previous
trials scored, so the search concentrates where results have been good. It can also stop weak
candidates part-way through, which the other two cannot do at all.

There is an older third option, **grid search**: list a few values for each hyperparameter and try
every combination. We do not use it in this course, and the reason is arithmetic rather than taste —
see the subsection below.

| | candidates come from | uses past results | good when |
|---|---|---|---|
| **Random** | draws from distributions | no | several hyperparameters, or the right order of magnitude is unknown |
| **Adaptive (Optuna)** | a model of what has scored well so far | yes | the budget is large, or each candidate is expensive |

Both are used below, and they are compared on the same problem at the same budget.

### What an exhaustive search costs

Grid search is the natural first idea, so it is worth seeing exactly why it is a poor way to spend a
budget. Its cost is a product:

$$\text{fits} = k \times \prod_{i=1}^{d} n_i$$

where:

- $\text{fits}$ — how many models are trained in total.
- $k$ — the number of cross-validation folds.
- $d$ — how many hyperparameters are being searched.
- $n_i$ — how many values are listed for the $i$-th hyperparameter, a whole number of 1 or more.

Three hyperparameters with three values each, at five folds, is $5 \times 27 = 135$ fits. Add a
fourth hyperparameter and it is 405; a fifth, 1,215. **Every question you add multiplies the bill**,
which is why grids stay small, and small grids are coarse.

Now count what those 135 fits actually explore. The 27 combinations contain only **three distinct
learning rates** — each one evaluated nine times, once per combination of the other two settings. If
the useful learning rate is not one of your three, no amount of budget elsewhere recovers it.

**A random search of 27 draws tries 27 distinct learning rates**, for the same 135 fits. When only a
few hyperparameters really matter — which is usual — that is a far better use of the same compute. So
for the rest of this session, candidates are drawn, never enumerated.

## When tuning pays: a model whose defaults are wrong

The tree's problem is not that trees are bad. It is that `DecisionTreeClassifier()` with no arguments
means *grow until every leaf is pure*, and nothing about that is suited to noisy customer data. The
defaults are simply far from what this dataset needs.

We give the search four hyperparameters:

- `max_depth` — how many questions deep the tree may go.
- `min_samples_leaf` — how many customers a leaf must hold. Large values stop the tree from carving
  out single people.
- `ccp_alpha` — the cost-complexity pruning strength: how much a branch must improve purity to be
  kept at all. Larger values cut more.
- `criterion` — how purity is measured, Gini or entropy.

> **In `scikit-learn`.** `RandomizedSearchCV(estimator, distributions, n_iter=..., scoring=..., cv=...)`
> draws `n_iter` candidates from the distributions and cross-validates each one. Inside a pipeline,
> a hyperparameter is named `step__hyperparameter`, so `model__max_depth` means "the estimator's
> `max_depth`". `best_estimator_` is refitted on the whole training set automatically.

The cell below draws 30 candidates. `randint` and `loguniform` are distributions to sample from, not
fixed lists — `loguniform` spreads its draws evenly across orders of magnitude, which is what a
strength parameter like `ccp_alpha` needs:

In [ ]:
tree_space = {
    "model__max_depth": randint(2, 20),
    "model__min_samples_leaf": randint(1, 120),
    "model__ccp_alpha": loguniform(1e-5, 1e-2),
    "model__criterion": ["gini", "entropy"],
}

tree_search = RandomizedSearchCV(
    make_model(DecisionTreeClassifier(random_state=RANDOM_STATE)),
    tree_space,
    n_iter=30,
    scoring="roc_auc",
    cv=5,
    random_state=RANDOM_STATE,
    n_jobs=-1,
)
tree_search

The search is built but not yet run. The cell below fits it, and reports how long the 150 fits took:

In [ ]:
started = time.perf_counter()
tree_search.fit(X_train, y_train)
finished = time.perf_counter()
print(f"30 candidates x 5 folds = {30 * 5} fits in {finished - started:.1f} seconds")

Every candidate is kept, not just the winner. The cell below shows the full table of results, best first — one row per candidate, with the settings it used and its score on each fold:

In [ ]:
pd.DataFrame(tree_search.cv_results_).sort_values(by="mean_test_score", ascending=False)

The cell below prints the winner's score:

In [ ]:
print(f"best cross-validated AUC: {tree_search.best_score_:.4f}")

And the cell below, the settings that produced it:

In [ ]:
print("best settings:")
pd.DataFrame([tree_search.best_params_])

**From 0.6594 to 0.8328.** Tuning bought **0.17 of AUC** — more than any modelling decision taken
anywhere in this course so far, and it changed nothing about the model except four numbers.

Read the settings it chose. `min_samples_leaf` of **90** is the one doing the work: no leaf may
describe fewer than ninety customers, so the tree cannot carve out individuals any more. `max_depth`
of 19 is almost irrelevant once that constraint is in place, which is a useful warning — a search
reports the best *combination*, and you cannot read the importance of one setting off it.

### When several candidates tie

The table above is sorted, so it invites you to take the top row. Look at the first three instead.

Their mean scores are **0.8328, 0.8322 and 0.8320** — a spread of 0.0008 — while the standard
deviation *within* each candidate, across the five folds, is about **0.010**. The gap between first
and third is roughly a twelfth of the noise. **The search has not shown that the winner is better; it
has shown that these three are indistinguishable.**

That matters, because the three are not equally attractive. Candidate 1 allows a depth of 19;
candidate 3 allows 6. When the evidence cannot separate two models, prefer the one that is **simpler,
smaller, faster or easier to explain** — that preference costs nothing here, and every one of those
properties has value later.

There is a trap in the same table, though, and it is worth naming: **do not break the tie with the
test set.** The test score is the one number you have left that nothing has chosen on.

## When tuning does not pay

Now the same treatment for gradient boosting. It starts at 0.8464, and its defaults are already
reasonable: shallow trees, a learning rate of 0.1, subsampling off.

We search five hyperparameters — the learning rate, the number of trees, the depth of each tree, the
fraction of rows each tree sees, and the minimum leaf size — over 27 candidates, the same budget
Optuna gets later.

The cell below runs it:

In [ ]:
boosting_space = {
    "model__learning_rate": loguniform(0.01, 0.5),
    "model__n_estimators": randint(30, 400),
    "model__max_depth": randint(2, 6),
    "model__subsample": [0.6, 0.8, 1.0],
    "model__min_samples_leaf": randint(1, 60),
}

random_search = RandomizedSearchCV(
    make_model(GradientBoostingClassifier(random_state=RANDOM_STATE)),
    boosting_space,
    n_iter=27,
    scoring="roc_auc",
    cv=5,
    random_state=RANDOM_STATE,
    n_jobs=-1,
)

The cell below runs it, and reports the time and the winning score next to the untuned one:

In [ ]:
started = time.perf_counter()
random_search.fit(X_train, y_train)
finished = time.perf_counter()

print(f"27 candidates x 5 folds = {27 * 5} fits in {finished - started:.1f} seconds")
print(f"best cross-validated AUC: {random_search.best_score_:.4f}   (untuned: 0.8464)")

**0.8489 against 0.8464.** The search moved the score by **0.0025**, against a fold-to-fold spread of
about 0.013 — twenty-seven candidates, a hundred and thirty-five fits, and the result is inside the
noise.

Put the two searches side by side, because together they are the lesson:

| | before | after | gain |
|---|---|---|---|
| Decision tree | 0.6594 | 0.8328 | **+0.1734** |
| Gradient boosting | 0.8464 | 0.8489 | +0.0025 |

**The gain from tuning lives in the distance between the defaults and what the data needs.** Library
defaults are chosen to be sane on typical data; for boosting they nearly were, and for an
unrestricted tree they were nowhere near. This is why "did you tune it?" is a weaker question than
"how far from sensible did it start?"

## The search space is where your knowledge goes

A search algorithm can only return something from the space you gave it. That makes the space — not
the algorithm — the place where knowing the model pays.

Two things go wrong most often:

- **The range misses the useful region.** If boosting needs a learning rate near 0.05 and your space
  starts at 0.5, no sampler will save you.
- **The scale is wrong.** Sampling a learning rate uniformly between 0.01 and 0.5 puts 80% of the
  draws above 0.1. Sampling it **log-uniformly** puts equal weight on 0.01–0.1 and 0.1–0.5, which is
  how a rate actually behaves.

This is demonstrated properly in Part 2, where the difference is large enough to see. Here, one line
about the space we used: `loguniform(0.01, 0.5)` for the rate and `randint(2, 6)` for the depth were
not arbitrary. Session 5 showed the validation curve peaking at a **small number of shallow trees**,
and the space was written around that.

## Optuna: a search that learns where to look

Random search does not learn. Every candidate is drawn without reference to how the previous ones
scored, so the hundredth draw is as uninformed as the first.

**Optuna** is a library for the second family in the table above: searches that do learn.
Before procedding further, it is convenient to familiarize with Optuna's vocabulary:

- A **study** is one optimization: an objective, a direction, and the record of everything tried.
- A **trial** is one evaluation of one set of hyperparameters.
- A **sampler** chooses the next trial's values. The default is **TPE** — *Tree-structured Parzen
  Estimator* — which models where good and bad scores have come from so far and samples where good
  ones are likelier.
- A **pruner** can stop a trial early, before it has finished, when it already looks hopeless.

The space is written **inside** the objective function, with `trial.suggest_*` calls. This is called a
*define-by-run* space, and it allows things a dictionary cannot express — a hyperparameter that only
exists when another one takes a particular value, for instance.

> **In `optuna`.** `optuna.create_study(direction="maximize", sampler=TPESampler(seed=...))`, then
> `study.optimize(objective, n_trials=...)`. The objective takes a `trial` and returns the number to
> optimize. Seed the sampler, or two runs of the same notebook will disagree.

The cell below gives Optuna the same space and the same budget as the random search — 27 trials — so
that the comparison is fair:

In [ ]:
def churn_objective(trial: optuna.Trial) -> float:
    """Cross-validated AUC of gradient boosting for one set of hyperparameters.

    Args:
        trial: The Optuna trial that proposes the hyperparameter values.

    Returns:
        The mean ROC AUC across five folds on the training data.
    """
    estimator = GradientBoostingClassifier(
        learning_rate=trial.suggest_float("learning_rate", 0.01, 0.5, log=True),
        n_estimators=trial.suggest_int("n_estimators", 30, 400),
        max_depth=trial.suggest_int("max_depth", 2, 5),
        subsample=trial.suggest_float("subsample", 0.6, 1.0),
        min_samples_leaf=trial.suggest_int("min_samples_leaf", 1, 60),
        random_state=RANDOM_STATE,
    )
    scores = cross_val_score(
        make_model(estimator), X_train, y_train, cv=5, scoring="roc_auc", n_jobs=-1
    )
    return scores.mean()

The objective is only a function until a study calls it. The cell below creates that study: the direction to optimize, and the sampler with its seed fixed so that the run reproduces:

In [ ]:
study = optuna.create_study(
    direction="maximize",
    sampler=optuna.samplers.TPESampler(seed=RANDOM_STATE)
)

The cell below runs the 27 trials and reports how long they took. With Optuna logging left on, each trial prints a line as it finishes:

In [ ]:
started = time.perf_counter()
study.optimize(churn_objective, n_trials=27)
finished = time.perf_counter()

print(f"27 trials in {finished - started:.1f} seconds")

The cell below prints the best score the study reached:

In [ ]:
print(f"best cross-validated AUC: {study.best_value:.4f}")

The cell below shows the settings behind that score:

In [ ]:
print("best settings:")
pd.DataFrame([study.best_params])

**Random search reached 0.8489 and Optuna 0.8486**, at 27 candidates each. The difference is 0.0003,
against a fold-to-fold spread of 0.013 — the two strategies found the same answer, and the simpler one
was marginally ahead.

**At this budget the three strategies are the same tool.** TPE needs enough trials to learn the shape
of the space before it can exploit it; with 27 it has barely started. Its advantage appears when
trials are expensive and numerous, which is exactly when you cannot demonstrate it in a lecture. What
you should take is the shape of the claim: *a smarter sampler pays when the budget is large enough for
it to learn something, and not before.*

### Reading what the study found

A study records every trial, so it can be asked questions that a single "best" number cannot answer.
Optuna ships its own plots for this.

The cell below draws the **optimization history**: each trial's score, and the best score so far. Look
at when the best-so-far line last moved — that says whether the budget was enough:

In [ ]:
fig = optuna_plots.plot_optimization_history(study)
fig.update_layout(width=800, height=500)
fig.show()

The best value is reached early and the line is then flat for most of the run: on this problem the
search had found what there was to find within the first handful of trials.

The cell below asks which hyperparameters mattered. Optuna measures this by how much the score varied
with each one across the trials:

In [ ]:
fig = optuna_plots.plot_param_importances(study)
fig.update_layout(width=800, height=500)
fig.show()

Read this as a statement about **the space we searched**, not about gradient boosting in general. A
hyperparameter whose range was narrow cannot look important however much it matters, and one that was
given a wide but useless range can look important simply because it moved the score around.

The cell below shows the trials themselves, one line per trial crossing every hyperparameter, coloured
by score. Regions where the good lines bunch together are the promising parts of the space:

In [ ]:
fig = optuna_plots.plot_parallel_coordinate(study)
fig.update_layout(width=800, height=500)
fig.show()

### Pruning: stop the hopeless trials early

**A word of warning about the name.** Earlier in this session, `ccp_alpha` pruned *a tree*: it cut
branches off one model. Here it is **trials** that are pruned — whole candidates abandoned part-way
through. Same word, unrelated mechanisms.

Most candidates in a search are bad, and normally you only learn that once the model is fully
trained. **Pruning** stops paying for them: while a candidate trains, it reports how it is doing, and
the pruner ends it as soon as it is clearly behind the candidates that came before.

Two conditions make this possible, and boosting meets both. The model must train **in steps** — one
tree at a time — and it must be able to report a score **while** it trains, not only at the end. That
timing is the whole trick: a pruner can only save work that has not been done yet.

> **In `optuna`.** `MedianPruner` stops a trial whose reported score is below the median of the
> previous trials at the same step; `n_warmup_steps` protects the first rounds, where every candidate
> looks bad. The reporting itself is one argument: the `optuna-integration` package provides
> `XGBoostPruningCallback(trial, "validation_0-auc")`, which watches the validation AUC after every
> boosting round and ends the trial when the pruner says so.

The cell below writes the objective for XGBoost — the same shape as before, plus that one argument.
Every candidate may use up to 500 boosting rounds, and the pruner decides how many it actually gets:

In [ ]:
X_fit, X_validation, y_fit, y_validation = train_test_split(
    X_train, y_train, test_size=0.2, stratify=y_train, random_state=RANDOM_STATE
)
X_fit_preproc = preprocessor.fit_transform(X_fit)
X_validation_preproc = preprocessor.transform(X_validation)


def xgb_objective(trial: optuna.Trial) -> float:
    """Validation AUC of XGBoost, reported after every round so that Optuna can prune the trial.

    Args:
        trial: The Optuna trial that proposes the hyperparameter values.

    Returns:
        The validation ROC AUC after the last round the trial was allowed to run.
    """
    model = xgb.XGBClassifier(
        n_estimators=500,
        learning_rate=trial.suggest_float("learning_rate", 0.01, 0.5, log=True),
        max_depth=trial.suggest_int("max_depth", 2, 8),
        subsample=trial.suggest_float("subsample", 0.5, 1.0),
        eval_metric="auc",
        random_state=RANDOM_STATE,
        n_jobs=-1,
        callbacks=[XGBoostPruningCallback(trial, "validation_0-auc")],
    )
    model.fit(
        X_fit_preproc, y_fit,
        eval_set=[(X_validation_preproc, y_validation)],
        verbose=False
    )
    return model.evals_result()["validation_0"]["auc"][-1]

The callback is the mechanism; the **pruner** is the policy that uses it. The cell below runs 20
trials with `MedianPruner`, then reports the time, the best score, and how many trials were stopped
before they finished:

In [ ]:
pruned_study = optuna.create_study(
    direction="maximize",
    sampler=optuna.samplers.TPESampler(seed=RANDOM_STATE),
    pruner=optuna.pruners.MedianPruner(n_startup_trials=5, n_warmup_steps=50),
)

started = time.perf_counter()
pruned_study.optimize(xgb_objective, n_trials=20)
pruned_seconds = time.perf_counter() - started

print(f"with pruning: 20 trials in {pruned_seconds:.1f} seconds")

The study keeps a record of every trial, including the ones it abandoned. The cell below counts them and reports how far they got before being stopped:

In [ ]:
stopped = [trial for trial in pruned_study.trials if trial.state == optuna.trial.TrialState.PRUNED]
rounds_used = [len(trial.intermediate_values) for trial in stopped]

print(f"best validation AUC: {pruned_study.best_value:.4f}")
print(f"trials stopped early: {len(stopped)} of 20")
print(f"they stopped after {int(np.mean(rounds_used))} rounds on average, out of 500")

That number means nothing on its own — 20 trials have to cost something. The cell below repeats the
identical search with `NopPruner`, a pruner that never prunes, so the only difference between the two
runs is whether trials may be stopped:

In [ ]:
full_study = optuna.create_study(
    direction="maximize",
    sampler=optuna.samplers.TPESampler(seed=RANDOM_STATE),
    pruner=optuna.pruners.NopPruner(),
)

started = time.perf_counter()
full_study.optimize(xgb_objective, n_trials=20)
full_seconds = time.perf_counter() - started

print(f"without pruning: 20 trials in {full_seconds:.1f} seconds")
print(f"  best validation AUC: {full_study.best_value:.4f}")
print(f"  every trial ran all 500 rounds")
print()
print(f"pruning saved {100 * (1 - pruned_seconds / full_seconds):.0f}% of the time, "
      f"and cost {full_study.best_value - pruned_study.best_value:.4f} of AUC")

**Pruning stopped 12 of the 20 trials and roughly halved the search time.** The seconds move from run
to run and from machine to machine, so read the two runs against each other rather than the digits.

The saving comes from where the stopped trials were stopped: after about **51 rounds out of 500**,
having done a tenth of their work.

Now the other side of the trade. The pruned run's best score is **0.8487**, against **0.8633** without
pruning — this time the pruner ended a candidate that would have come good. A trial that is behind
after 50 rounds is usually behind at 500, but *usually* is not *always*: a small learning rate looks
terrible early and can win in the end. `n_warmup_steps` exists to limit exactly this, and **pruning
stays a trade — cheaper searches, and a real risk of throwing away the winner.**

The cell below draws every trial's progress, one line each. The lines that stop short are the ones the
pruner ended:

In [ ]:
fig = optuna_plots.plot_intermediate_values(pruned_study)
fig.update_layout(width=800, height=500)
fig.show()

## Nested cross-validation: what did the *procedure* achieve?

Here is the problem the whole session has been walking towards.

The search evaluated 27 candidates and reported the best cross-validated score it saw: 0.8483. But
that number was the **maximum of 27 noisy estimates**, and the maximum of a set of noisy numbers is
optimistic. Part of what the winner scored is real quality, and part is luck on those particular
folds. Reporting it as "what this model achieves" is a softer version of reporting a test score you
selected on.

**Nested cross-validation** answers the question properly. It has two loops:

- the **inner** loop runs the whole search on the training part of each outer fold, and picks a
  winner;
- the **outer** loop scores that winner on the outer fold, which the search never saw.

What comes out is not a hyperparameter setting. It is an estimate of **what your tuning procedure
produces when it meets new data** — search included. Its cost:

$$\text{fits} = k_{\text{outer}} \times n \times k_{\text{inner}}$$

where:

- $k_{\text{outer}}$ — the number of outer folds, which give the honest estimate.
- $n$ — the number of candidates the inner search evaluates.
- $k_{\text{inner}}$ — the number of folds inside the search.

Five outer folds with a ten-candidate search at three inner folds is $5 \times 10 \times 3 = 150$
fits. The cell below runs exactly that, then runs the same search once on the whole training set for
comparison:

In [ ]:
inner_search = RandomizedSearchCV(
    make_model(GradientBoostingClassifier(random_state=RANDOM_STATE)),
    boosting_space,
    n_iter=10,
    scoring="roc_auc",
    cv=3,
    random_state=RANDOM_STATE,
    n_jobs=-1,
)

started = time.perf_counter()
nested_scores = cross_val_score(inner_search, X_train, y_train, cv=5, scoring="roc_auc")
nested_seconds = time.perf_counter() - started

inner_search.fit(X_train, y_train)
selected = inner_search.best_estimator_
test_auc = roc_auc_score(y_test, selected.predict_proba(X_test)[:, 1])

print(f"nested CV (5 x 10 x 3 = 150 fits) in {nested_seconds:.1f} seconds")
print(f"  honest estimate of the procedure: {nested_scores.mean():.4f} (sd {nested_scores.std():.4f})")
print(f"  the search's own best score:      {inner_search.best_score_:.4f}")
print(f"  that winner on the test set:      {test_auc:.4f}")

Three numbers, and the order matters more than the values.

- **The nested estimate is 0.8486.** This is what the sentence "we tune gradient boosting with a
  ten-candidate random search" is worth on this data.
- **The search's own best score is also 0.8486**, and **the winner scores 0.8478 on the test set.**

Here the three agree to within 0.001, which is a real result and a mild one: with only ten candidates
and mild noise, the optimism is small. It grows with the number of candidates — search a thousand and
the winner's score becomes substantially a report on luck.

**When is nesting worth its cost?** When the claim you are making is about the procedure — "tuning
buys us this much" — or when candidates are many and differences are small. When you simply need a
model to ship, a single search plus one honest test-set evaluation is enough, which is what the rest
of this course does.

# Part 2 — The same procedure on a regression problem

Everything so far was one dataset and one task. Session 5 ended by showing that the ranking of model
families flips between datasets; the question here is narrower and different: **does what we learned
about tuning hold when the task changes?**

We return to the **Ames housing** data from sessions 2 and 3: 1,460 houses, 79 features, and
`SalePrice` as the target.

**One change to the target, and the reason for it.** House prices are heavily skewed — most houses
sell around 150,000 and a few sell for over 700,000. RMSE in dollars is then dominated by the handful
of expensive houses, and it moves so much between folds that no comparison in this session would be
readable. We therefore predict the **natural logarithm of the price**. Two consequences:

- the error is reported in log units, and a log RMSE of about 0.13 means the typical prediction is out
  by roughly **13%** of the house's price, which is easier to act on than a figure in dollars;
- large and small houses now count equally, which is usually what a price model should want.

The cell below loads Ames, takes the logarithm of the target, and builds a preparation pipeline —
median imputation and scaling for the numbers, most-frequent imputation and one-hot encoding for the
text columns:

In [ ]:
ames = pd.read_csv("data/ames_housing.csv")
ames_y = np.log(ames["SalePrice"])
ames_X = ames.drop(columns=[column for column in ["SalePrice", "Id"] if column in ames.columns])

ames_numeric = ames_X.select_dtypes("number").columns.tolist()
ames_categorical = [column for column in ames_X.columns if column not in ames_numeric]
ames_preprocessor = ColumnTransformer(
    transformers=[
        ("num", Pipeline([("impute", SimpleImputer(strategy="median")), ("scale", StandardScaler())]),
         ames_numeric),
        ("cat", Pipeline([("impute", SimpleImputer(strategy="most_frequent")),
                          ("encode", OneHotEncoder(handle_unknown="ignore", sparse_output=False))]),
         ames_categorical),
    ]
)


def make_ames_model(estimator) -> Pipeline:
    """Chain the Ames preparation to a regressor.

    Args:
        estimator: Any scikit-learn regressor.

    Returns:
        A pipeline that prepares the columns and then applies the regressor.
    """
    return Pipeline([("prep", ames_preprocessor), ("model", estimator)])


ames_X_train, ames_X_test, ames_y_train, ames_y_test = train_test_split(
    ames_X, ames_y, test_size=0.25, random_state=RANDOM_STATE
)

print(f"{ames_X.shape[0]} houses, {ames_X.shape[1]} features "
      f"({len(ames_numeric)} numeric, {len(ames_categorical)} text)")
print(f"log price ranges from {ames_y.min():.2f} to {ames_y.max():.2f}")

The cell below establishes the two starting points, exactly as Part 1 did: a linear model and
gradient boosting at its defaults. **Lower is better** now, because the score is an error:

In [ ]:
def cv_rmse(estimator, label: str) -> dict:
    """Cross-validate one regressor on the Ames training data.

    Args:
        estimator: Any scikit-learn regressor, without preprocessing.
        label: Name to report the result under.

    Returns:
        The label, the mean RMSE in log units across five folds and its standard deviation.
    """
    scores = -cross_val_score(
        make_ames_model(estimator), ames_X_train, ames_y_train, cv=5,
        scoring="neg_root_mean_squared_error", n_jobs=-1,
    )
    return {"model": label, "CV RMSE (log)": scores.mean(), "sd": scores.std()}


ridge_row = cv_rmse(Ridge(alpha=10), "Ridge (alpha=10)")
defaults_row = cv_rmse(
    GradientBoostingRegressor(random_state=RANDOM_STATE), "Gradient boosting (defaults)"
)

pd.DataFrame([ridge_row, defaults_row]).round(4)

Now the search itself, in the same two steps as Part 1: state the space, then spend the budget in it.
This space is written around what session 5 showed about boosting — many shallow trees, a small
learning rate — and 30 candidates are drawn from it:

In [ ]:
sensible_space = {
    "model__learning_rate": loguniform(0.01, 0.5),
    "model__n_estimators": randint(100, 800),
    "model__max_depth": randint(2, 6),
    "model__subsample": uniform(0.6, 0.4),
    "model__min_samples_leaf": randint(1, 30),
}

sensible_search = RandomizedSearchCV(
    make_ames_model(GradientBoostingRegressor(random_state=RANDOM_STATE)),
    sensible_space,
    n_iter=30,
    scoring="neg_root_mean_squared_error",
    cv=5,
    random_state=RANDOM_STATE,
    n_jobs=-1,
)

The cell below runs it, and reports the error of the best candidate it found:

In [ ]:
started = time.perf_counter()
sensible_search.fit(ames_X_train, ames_y_train)
finished = time.perf_counter()

print(f"30 candidates x 5 folds = {30 * 5} fits in {finished - started:.1f} seconds")
print(f"best CV RMSE (log): {-sensible_search.best_score_:.4f}   (defaults: {defaults_row['CV RMSE (log)']:.4f})")

**The same search, in a space that is wrong.** The cell below asks for a large learning rate, very
few trees, and deep ones — the opposite of what boosting wants. Nothing else changes: the same
algorithm, the same 30 candidates, the same folds, the same seed:

In [ ]:
bad_space = {
    "model__learning_rate": uniform(0.5, 0.5),
    "model__n_estimators": randint(5, 40),
    "model__max_depth": randint(8, 14),
    "model__min_samples_leaf": randint(1, 3),
}

bad_search = RandomizedSearchCV(
    make_ames_model(GradientBoostingRegressor(random_state=RANDOM_STATE)),
    bad_space,
    n_iter=30,
    scoring="neg_root_mean_squared_error",
    cv=5,
    random_state=RANDOM_STATE,
    n_jobs=-1,
)

The cell below runs that one, and prints the same two numbers for comparison:

In [ ]:
started = time.perf_counter()
bad_search.fit(ames_X_train, ames_y_train)
finished = time.perf_counter()

print(f"30 candidates x 5 folds = {30 * 5} fits in {finished - started:.1f} seconds")
print(f"best CV RMSE (log): {-bad_search.best_score_:.4f}   (defaults: {defaults_row['CV RMSE (log)']:.4f})")

The cell below puts the four results in one table, best first:

In [ ]:
ames_results = pd.DataFrame(
    [
        ridge_row,
        defaults_row,
        {"model": "Boosting, sensible space", "CV RMSE (log)": -sensible_search.best_score_, "sd": None},
        {"model": "Boosting, bad space", "CV RMSE (log)": -bad_search.best_score_, "sd": None},
    ]
).sort_values("CV RMSE (log)")

ames_results.round(4)

**The sensible space reaches 0.1331; the bad space reaches 0.1732.** Same algorithm, same budget, same
data, same seed — and a result **worse than the untuned defaults** (0.1422).

That is the section's whole claim, measured: **a search cannot rescue a space that does not contain
good models.** The bad space asked for a large learning rate with very few, very deep trees, which
is the opposite of what boosting wants, and thirty draws inside it found the best of a bad set.

Compare the two searches' costs as well: the bad space ran in about a quarter of the time, because
models with 5 to 40 trees are quick to fit. **A search that finishes suspiciously fast is worth
looking at.**

### Did the improvement survive?

The sensible search improved the cross-validated error from 0.1422 to 0.1331 — a gain of about 6%.
Part 1 said that such a gain is partly real and partly luck on those folds. Here we can check, because
a test set has been waiting since the split.

The cell below scores three models on it: the tuned boosting, boosting at its defaults, and Ridge:

In [ ]:
tuned_ames = sensible_search.best_estimator_

ames_defaults = make_ames_model(GradientBoostingRegressor(random_state=RANDOM_STATE))
ames_defaults.fit(ames_X_train, ames_y_train)

ames_ridge = make_ames_model(Ridge(alpha=10))
ames_ridge.fit(ames_X_train, ames_y_train)

tuned_error = root_mean_squared_error(ames_y_test, tuned_ames.predict(ames_X_test))
defaults_error = root_mean_squared_error(ames_y_test, ames_defaults.predict(ames_X_test))
ridge_error = root_mean_squared_error(ames_y_test, ames_ridge.predict(ames_X_test))

pd.DataFrame(
    [
        {"model": "Gradient boosting (tuned)", "test RMSE (log)": tuned_error},
        {"model": "Gradient boosting (defaults)", "test RMSE (log)": defaults_error},
        {"model": "Ridge (alpha=10)", "test RMSE (log)": ridge_error},
    ]
).assign(**{"typical error (%)": lambda frame: (100 * (np.exp(frame["test RMSE (log)"]) - 1)).round(1)}).round(4)

**The tuned model scores 0.1268 on the test set. The untuned one scores 0.1266.**

The six percent that the search found in cross-validation did **not** survive contact with new data.
The search was not broken and the code is not wrong: it did exactly what it was asked, which was to
find the candidate that scored best on those five folds. Part of that score was the candidate being
better, and part was it suiting those folds — and only the first part transfers.

Both boosted models beat Ridge (0.1331), so the choice of model family was worth more than any
tuning. And a typical error of about **13.5%** of a house's price is the number to quote, not the
cross-validated one.

The cell below runs nested cross-validation here too, so the procedure gets an honest estimate rather
than an inherited suspicion:

In [ ]:
ames_inner = RandomizedSearchCV(
    make_ames_model(GradientBoostingRegressor(random_state=RANDOM_STATE)),
    sensible_space, n_iter=10, scoring="neg_root_mean_squared_error", cv=3,
    random_state=RANDOM_STATE, n_jobs=-1,
)
started = time.perf_counter()
ames_nested = -cross_val_score(
    ames_inner, ames_X_train, ames_y_train, cv=5, scoring="neg_root_mean_squared_error"
)

print(f"nested CV in {time.perf_counter() - started:.1f} seconds")
print(f"  honest estimate of the procedure: {ames_nested.mean():.4f} (sd {ames_nested.std():.4f})")
print(f"  the flat search reported:         {-sensible_search.best_score_:.4f}")
print(f"  defaults, cross-validated:        {defaults_row['CV RMSE (log)']:.4f}")

**The nested estimate is 0.1350, against the search's own 0.1331 and the defaults' 0.1422.**

Read the three together and the story is consistent from both halves of the session. The tuning
procedure is worth something here — 0.1350 is better than 0.1422 — but **less than the search claimed
for itself**, and the gap between 0.1331 and 0.1350 is exactly the optimism that nesting was built to
remove.

# Wrap-up

## What this session established

- **Tuning optimizes a cross-validated score on the training data.** The test set is opened once, at
  the end. Anything else lets the search consume the evidence you were going to report.
- **The gain from tuning is the distance between the defaults and what the data needs.** An
  unrestricted tree went from 0.6594 to 0.8328 — the largest single gain in this course. Gradient
  boosting went from 0.8464 to 0.8489, which is inside the noise.
- **Exhaustive search multiplies**: $k \times \prod n_i$. A three-by-three-by-three grid at five folds
  is 135 fits, and buys only **three** distinct learning rates. The same 135 fits spent on random
  draws buy 27 — which is why candidates are drawn, not enumerated.
- **When candidates tie, the search has not chosen for you.** The top three tree candidates sat within
  0.0008 of each other against a within-candidate spread of 0.010, so the tie was broken on
  simplicity — and never on the test set.
- **A smarter sampler needs room to learn.** At 27 trials, random search reached 0.8489 and Optuna's
  TPE 0.8486: the same answer twice, with the simpler strategy marginally ahead.
- **Pruning buys trials and risks late bloomers.** It stopped 13 of 20 trials, halved the time, and
  found a slightly worse best (0.8564 against 0.8629).
- **The space carries your knowledge of the model.** On Ames, the same algorithm and budget gave
  0.1331 in a sensible space and 0.1732 in a plausible but wrong one — worse than not tuning at all.
- **The search's own best score is optimistic**, because it is the maximum of many noisy estimates.
  Nested cross-validation estimates the procedure instead: 0.8486 on churn, and 0.1350 on Ames against
  the flat search's 0.1331.
- **A cross-validated gain need not survive.** On Ames, tuning improved cross-validation by 6% and the
  test error not at all.

## What this session did not do

**We never asked why a model predicts what it predicts.** Session 5 left a feature ranking flagged as
unreliable, and this session has added tuned models that are harder to read than the ones they
replaced. **Session 7** takes up interpretability and trust: global and local explanations, what goes
wrong when features are correlated, and the model card as the thing you hand over.

**We searched tiny budgets.** Every search here was 20 to 30 candidates so that the notebook runs in
minutes. Real tuning runs are thousands of trials, run in parallel, and that is where TPE and pruning
earn their place — the mechanisms are the same ones you have just used.